# Comparison of DFT solvers

We compare four different approaches for solving the DFT minimisation problem,
namely a density-based SCF, a potential-based SCF, direct minimisation and Newton.

First we setup our problem

In [1]:
using AtomsBuilder
using DFTK
using LinearAlgebra
using PseudoPotentialData

pseudopotentials = PseudoFamily("dojo.nc.sr.pbesol.v0_4_1.standard.upf")
model = model_DFT(bulk(:Si); functionals=PBEsol(), pseudopotentials)
basis = PlaneWaveBasis(model; Ecut=5, kgrid=[3, 3, 3])

# Convergence we desire in the density
tol = 1e-6

1.0e-6

## Density-based self-consistent field

In [2]:
scfres_scf = self_consistent_field(basis; tol);

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -8.397779608273                   -0.89    5.2   28.2ms
  2   -8.400244030170       -2.61       -1.74    1.0   19.4ms
  3   -8.400398262944       -3.81       -2.92    1.2   19.9ms
  4   -8.400427745053       -4.53       -2.89    3.2   25.5ms
  5   -8.400427982347       -6.62       -3.07    1.0   19.3ms
  6   -8.400428147663       -6.78       -5.01    1.0   19.2ms
  7   -8.400428151898       -8.37       -4.49    3.5   59.1ms
  8   -8.400428152202       -9.52       -5.81    1.5   21.3ms
  9   -8.400428152209      -11.14       -5.87    2.0   22.5ms
 10   -8.400428152209      -12.42       -7.55    1.0   20.3ms


## Potential-based SCF

In [3]:
scfres_scfv = DFTK.scf_potential_mixing(basis; tol);

n     Energy            log10(ΔE)   log10(Δρ)   α      Diag   Δtime
---   ---------------   ---------   ---------   ----   ----   ------
  1   -8.397807762596                   -0.90           5.0    827ms
  2   -8.400380951452       -2.59       -1.79   0.80    2.2    502ms
  3   -8.400422329157       -4.38       -3.03   0.80    1.0    215ms
  4   -8.400428113950       -5.24       -3.45   0.80    2.8   21.7ms
  5   -8.400428148653       -7.46       -4.75   0.80    1.0   16.9ms
  6   -8.400428152198       -8.45       -5.60   0.80    2.8   21.9ms
  7   -8.400428152208      -10.98       -6.31   0.80    1.8   18.6ms


## Direct minimization

In [4]:
scfres_dm = direct_minimization(basis; tol);

┌ Warning: x_tol is deprecated. Use x_abstol or x_reltol instead. The provided value (-1) will be used as x_abstol.
└ @ Optim ~/.julia/packages/Optim/7krni/src/types.jl:110
┌ Warning: f_tol is deprecated. Use f_abstol or f_reltol instead. The provided value (-1) will be used as f_reltol.
└ @ Optim ~/.julia/packages/Optim/7krni/src/types.jl:120
n     Energy            log10(ΔE)   log10(Δρ)   Δtime
---   ---------------   ---------   ---------   ------
  1   +1.157440563841                   -1.05    3.31s
  2   -1.460836392447        0.42       -0.70    140ms
  3   -4.356448959565        0.46       -0.38   44.1ms
  4   -5.925942451647        0.20       -0.48   44.0ms
  5   -7.442599339454        0.18       -0.71   83.4ms
  6   -7.899307231238       -0.34       -1.41   32.8ms
  7   -8.198200128899       -0.52       -1.69   32.8ms
  8   -8.266258159253       -1.17       -1.88   33.0ms
  9   -8.315935034525       -1.30       -2.23   32.8ms
 10   -8.337164314041       -1.67       -2.07   57

## Newton algorithm

Start not too far from the solution to ensure convergence:
We run first a very crude SCF to get close and then switch to Newton.

In [5]:
scfres_start = self_consistent_field(basis; tol=0.5);

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -8.397804753346                   -0.90    5.0   26.8ms


Remove the virtual orbitals (which Newton cannot treat yet)

In [6]:
ψ = DFTK.select_occupied_orbitals(basis, scfres_start.ψ, scfres_start.occupation).ψ
scfres_newton = newton(basis, ψ; tol);

n     Energy            log10(ΔE)   log10(Δρ)   Δtime
---   ---------------   ---------   ---------   ------
  1   -8.400427967029                   -1.78    8.28s
  2   -8.400428152209       -6.73       -4.02    3.10s
  3   -8.400428152209      -14.75       -7.79    139ms


## Comparison of results

In [7]:
println("|ρ_newton - ρ_scf|  = ", norm(scfres_newton.ρ - scfres_scf.ρ))
println("|ρ_newton - ρ_scfv| = ", norm(scfres_newton.ρ - scfres_scfv.ρ))
println("|ρ_newton - ρ_dm|   = ", norm(scfres_newton.ρ - scfres_dm.ρ))

|ρ_newton - ρ_scf|  = 1.090117460291037e-7
|ρ_newton - ρ_scfv| = 1.559394777430082e-7
|ρ_newton - ρ_dm|   = 2.207477395800833e-6
